# Use-case: WWTD-2025 (What Would Trump Do?)

Generate a forecasting dataset about Trump's actions, decisions, and statements using the LightningRod SDK. This example showcases dataset generation, preparation with SDK utils, and training results from our experiments—including evaluation with and without context.

In [2]:
%pip install lightningrod-ai python-dotenv pandas

from IPython.display import clear_output
clear_output()

from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

In [3]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Build the pipeline

Configure the pipeline with domain-specific instructions and examples for Trump-related forecasting.

In [4]:
instructions = """
Generate binary forecasting questions about Trump's actions, decisions, positions, and statements.
Questions should be diverse, related to the content, and should evenly cover the full range from very likely to very unlikely.
Horizon: outcomes should be known within 2 months of the question date, and may be known much sooner.
Criteria: binary outcome, exact dates, self-contained, verifiable via web search, newsworthy.
"""

good_examples = [
    "Will Trump impose 25% tariffs on all goods from Canada by February 1, 2025?",
    "Will Trump issue pardons to January 6 defendants within his first week in office?",
    "Will Pete Hegseth be confirmed as Secretary of Defense by February 15, 2025?",
    "Will Trump sign an executive order to keep TikTok operational in the US by January 31, 2025?",
    "Will Kash Patel be confirmed as FBI Director by March 1, 2025?",
]

bad_examples = [
    "Will Trump do something controversial? (too vague)",
    "Will Trump be in the news? (obvious)",
    "Will tariffs be imposed? (needs specifics)",
]

In [5]:
from lightningrod import (
    BinaryAnswerType,
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    NewsContextGenerator,
    WebSearchLabeler,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    seed_generator=NewsSeedGenerator(
        start_date=datetime(2025, 1, 1),
        end_date=datetime(2026, 1, 1),
        interval_duration_days=7,
        search_query=[
            "Donald Trump domestic policy agenda",
            "Donald Trump trade and tariff actions",
            "Donald Trump foreign policy decisions",
            "Donald Trump interviews and press appearances",
            "Donald Trump lawsuits and court rulings",
        ],
        articles_per_search=10,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        answer_type=answer_type,
        questions_per_seed=20,
    ),
    context_generators=[
        NewsContextGenerator(
            articles_per_query=3,
            num_search_queries=1,
            num_articles=5,
        )
    ],
    labeler=WebSearchLabeler(answer_type=answer_type),
)

## Run the pipeline

This will collect news articles, generate questions, and find answers. Use `max_questions` to limit the run for testing.

In [7]:
dataset = lr.transforms.run(pipeline, max_questions=1000, name="WWTD-2025")

samples = dataset.download()
pct = (dataset.valid_count() / len(samples) * 100) if samples else 0
print(f"Generated {len(samples)} samples ({pct:.1f}% valid)")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Warning                                                                                                     │
│                                                                                                                 │
│  Estimated cost ($41.88) exceeds current balance ($0.00). Consider adding credits before running this job.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Generated 364 samples (55.5% valid)


## Prepare the dataset

Use SDK utils to filter valid samples, deduplicate, and split into train/test sets. WWTD uses a 80/20 random split (`test_fraction=0.2`). The experiment also evaluated models with and without context—we show the with-context path here.

In [9]:
from lightningrod.utils import deduplicate_samples, test_train_split, filter_samples, add_rl_training_fields, render_sample, flatten_samples

valid_samples = filter_samples(samples, drop_missing_context=False)

samples_prep = deduplicate_samples(valid_samples)
print(f"Valid+deduped: {len(samples_prep)}")

train, test = test_train_split(samples_prep, test_fraction=0.2, filter_leaky_train=True)

def _yes_count(sample_list):
    return sum(
        1 for s in sample_list
        if s.label and str(s.label.label).lower() in ("1", "true", "yes")
    )

for name, data in [("Train", train), ("Test", test)]:
    yes = _yes_count(data)
    if len(data) > 0:
        print(f"{name}: {len(data)} rows, {yes/len(data)*100:.1f}% yes")
    else:
        print(f"{name}: no rows")

add_rl_training_fields(train, answer_type)
_ = add_rl_training_fields(test, answer_type)

Valid+deduped: 202
Train: 139 rows, 25.2% yes
Test: 41 rows, 46.3% yes


In [10]:
def _display_head(data, name, n=5):
    rows = flatten_samples(data[:n])
    if not rows:
        print(f"{name}: no rows")
        return
    df = pd.DataFrame(rows)
    cols = [
        "question_text", "label", "correct_answer",
        "question_prediction_date", "question_date_close", "label_resolution_date",
        "answer_type", "answer_parser_type", "reward_function_type",
        "label_confidence",
    ]
    display_cols = [c for c in cols if c in df.columns]
    rename = {
        "question_text": "Question",
        "label": "Answer",
        "label_confidence": "Confidence",
        "correct_answer": "Correct",
        "question_prediction_date": "Prediction Date",
        "question_date_close": "Close Date",
        "label_resolution_date": "Resolution Date",
        "answer_type": "Type",
        "answer_parser_type": "Parser",
        "reward_function_type": "Reward",
    }
    print(f"{name} (head):")
    display(df[display_cols].rename(columns=rename))

_display_head(train, "Train")
_display_head(test, "Test")

print("Sample prompt (first train example):")
render_sample(train[0], answer_type=answer_type)[0]["content"]

Train (head):


,Question,Answer,Correct,Prediction Date,Close Date,Resolution Date,Type,Parser,Reward,Confidence
0,Will President Trump issue an executive order ...,0,0.0,2025-01-10T00:00:00,2025-02-16T00:00:00,2025-02-15T00:00:00,binary,binary,binary_log_score,1.0
1,Will the Congressional Budget Office (CBO) pub...,1,1.0,2025-01-10T00:00:00,2025-03-16T00:00:00,2025-03-05T00:00:00,binary,binary,binary_log_score,1.0
2,Will the House Budget Committee report a recon...,0,0.0,2025-01-10T00:00:00,2025-03-11T00:00:00,2025-03-11T00:00:00,binary,binary,binary_log_score,1.0
3,Will the Trump administration formally propose...,1,1.0,2025-01-10T00:00:00,2025-03-02T00:00:00,2025-01-28T00:00:00,binary,binary,binary_log_score,0.9
4,Will the House of Representatives pass a budge...,0,0.0,2025-01-10T00:00:00,2025-03-01T00:00:00,2025-03-01T00:00:00,binary,binary,binary_log_score,1.0


Test (head):


,Question,Answer,Correct,Prediction Date,Close Date,Resolution Date,Type,Parser,Reward,Confidence
0,Will President Trump publicly announce a new t...,0,0.0,2025-09-09T00:00:00,2025-10-15T00:00:00,2025-10-15T00:00:00,binary,binary,binary_log_score,1.0
1,Will the US Supreme Court announce whether it ...,1,1.0,2025-09-09T00:00:00,2025-09-15T00:00:00,2025-09-09T00:00:00,binary,binary,binary_log_score,1.0
2,Will President Trump sign a new executive orde...,1,1.0,2025-09-09T00:00:00,2025-10-31T00:00:00,2025-10-01T00:00:00,binary,binary,binary_log_score,1.0
3,Will the US Supreme Court hear oral arguments ...,1,1.0,2025-09-09T00:00:00,2025-11-10T00:00:00,2025-11-05T00:00:00,binary,binary,binary_log_score,1.0
4,Will the Supreme Court issue a stay on the fed...,0,0.0,2025-09-10T00:00:00,2025-10-01T00:00:00,2025-10-01T00:00:00,binary,binary,binary_log_score,1.0


Sample prompt (first train example):


'QUESTION:\nWill President Trump issue an executive order or formal memorandum specifically directing the Department of Health and Human Services (HHS) to implement work requirements for Medicaid by February 15, 2025?\n\nTODAY\'S DATE:\n2025-01-10\n\nRESOLUTION CRITERIA:\nThe question resolves to Yes if an official Executive Order or Presidential Memorandum is published on the Federal Register or the White House website by February 15, 2025, that explicitly instructs HHS to facilitate or mandate work requirements for Medicaid recipients.\n\nCLOSE DATE:\n2025-02-16\n\nCONTEXT:\nNEWS:\nRecent news articles relevant to this question:\n\n---\nARTICLES\n[1] Medicaid and a Second Trump Administration (published on 2024-12-11 by milbank.org) (relevance: 5.0)\nSummary: A second Trump administration will likely attack Medicaid for funding tax cuts, leveraging decades-old tropes against beneficiaries. Legislative reforms face reduced guardrails, though governors\' roles provide some leverage. Pr

## Model Training Results

We fine-tuned a forecasting model via RL on a dataset of 2,790 questions, surpassing GPT-5 performance.

<br>

![Brier Skill Score](https://huggingface.co/datasets/LightningRodLabs/WWTD-2025/resolve/main/brier_skill_score.png)

<br>

**For more details on methods, results, and data, visit the HuggingFace links below:**
- **[Trump-Forecaster Model](https://huggingface.co/LightningRodLabs/Trump-Forecaster)**
- **[Trump-Forecaster Dataset](https://huggingface.co/datasets/LightningRodLabs/WWTD-2025)**

<br>

---

<br>

🚀 **Coming Soon:** Seamlessly generate datasets, fine-tune, and evaluate your own forecasting models end-to-end on the Lightningrod platform.
 
👉 [Sign up to get early access and updates.](https://lightningrod.ai/)